In [3]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [4]:
# from huggingface_hub import snapshot_download

# snapshot_download(
#     repo_id="imvladikon/hebrew_speech_coursera", 
#     repo_type="dataset", local_dir="./hebrew_speech_coursera", allow_patterns="data/*.parquet")

In [5]:
files = glob('hebrew_speech_coursera/*/*.parquet')
len(files)

18

In [6]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in files:
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in tqdm(range(len(df))):
            t = df['sentence'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            if not os.path.exists(audio_filename):
                b = df['audio'].iloc[i]['bytes']
                audio_np, sr = sf.read(io.BytesIO(b))
                if audio_np.ndim > 1:
                    audio_np = audio_np.mean(axis=1)
                if audio_np.shape[0] < 10000:
                    continue
                sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}"
            })
        
    return data

In [7]:
data = multiprocessing(files, loop, cores = 5)

100%|██████████| 1450/1450 [00:00<00:00, 86333.18it/s]


In [9]:
len(data)

25382

In [10]:
audio_files = [d['audio_filename'] for d in data]

with open('hebrew_speech_coursera-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [11]:
with open('hebrew_speech_coursera.json', 'w') as fopen:
    json.dump(data, fopen)

In [9]:
# !zip -rq hebrew_speech_coursera_audio.zip hebrew_speech_coursera_audio
# !hf upload malaysia-ai/Multilingual-TTS hebrew_speech_coursera_audio.zip --repo-type=dataset